# Async Transport

> Async equivalents of the core streaming functions using `httpx`.
> Install with `pip install mcp-ski-resort[async]`.

In [ ]:
#| default_exp astream

In [ ]:
#| export
try:
    import httpx
except ImportError:
    raise ImportError(
        "httpx is required for async support. "
        "Install it with: pip install mcp-ski-resort[async]"
    )

import json
import time
from collections.abc import AsyncIterator, Callable
from typing import Any

from mcp_ski_resort.core import (
    SnowflakeSession,
    default_session,
    build_agent_run_payload,
    normalize_event,
    AgentResult,
    _apply_normalized_event,
    _finalize_agent_result,
    _default_reporter,
)


## Async Streaming

These functions mirror the sync API in `core.py` but use `httpx.AsyncClient`
for non-blocking I/O. All functions accept an optional `client` parameter
for connection reuse (e.g., from a FastAPI lifespan).

In [ ]:
#| export
async def async_create_thread(
    session: SnowflakeSession | None = None,
    origin_application: str = "mcp_ski_resort",
    client: httpx.AsyncClient | None = None,
) -> str:
    """Create a Cortex conversation thread (async). Returns the ``thread_id`` string.

    Raises ``KeyError`` if the response contains neither ``thread_id``
    nor ``id``.
    """
    s = session or default_session()
    _client = client or httpx.AsyncClient(timeout=30)
    try:
        resp = await _client.post(
            f"{s.host}/api/v2/cortex/threads",
            headers=s.get_headers(),
            json={"origin_application": origin_application},
        )
        resp.raise_for_status()
        data = resp.json()
        tid = data.get("thread_id") or data.get("id")
        if tid is None:
            raise KeyError(f"Thread response missing thread_id and id: {list(data.keys())}")
        return str(tid)
    finally:
        if client is None:
            await _client.aclose()


async def async_stream_agent_sse(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
    client: httpx.AsyncClient | None = None,
) -> AsyncIterator[dict]:
    """POST to the Cortex Agent :run endpoint and yield parsed SSE events (async).

    Each yielded dict has ``{"event": str, "data": dict}``.
    Malformed JSON in SSE data lines yields a ``parse_error`` event instead
    of being silently dropped.
    """
    s = session or default_session()
    endpoint = (
        f"{s.host}/api/v2/databases/{s.database}"
        f"/schemas/{s.agent_schema}/agents/{agent_name}:run"
    )
    payload = build_agent_run_payload(question, history, thread_id, parent_message_id)

    _client = client or httpx.AsyncClient(timeout=120)
    try:
        async with _client.stream(
            "POST", endpoint,
            headers=s.get_headers(accept="text/event-stream"),
            json=payload,
        ) as resp:
            resp.raise_for_status()

            current_event: str | None = None
            data_buffer: list[str] = []

            async for raw_line in resp.aiter_lines():
                line = raw_line

                if not line:
                    if data_buffer and current_event is not None:
                        joined = "\n".join(data_buffer)
                        if joined == "[DONE]":
                            yield {"event": "done", "data": {}}
                            return
                        try:
                            data = json.loads(joined)
                            yield {"event": current_event, "data": data}
                        except json.JSONDecodeError as exc:
                            yield {
                                "event": "parse_error",
                                "data": {"raw_event": current_event, "raw_data": joined[:500], "error": str(exc)},
                            }
                    current_event = None
                    data_buffer = []
                    continue

                if line.startswith("event:"):
                    current_event = line[6:].strip()
                    continue

                if line.startswith("data:"):
                    data_buffer.append(line[5:].strip())
                    continue

            if data_buffer and current_event is not None:
                joined = "\n".join(data_buffer)
                if joined == "[DONE]":
                    yield {"event": "done", "data": {}}
                    return
                try:
                    data = json.loads(joined)
                    yield {"event": current_event, "data": data}
                except json.JSONDecodeError as exc:
                    yield {
                        "event": "parse_error",
                        "data": {"raw_event": current_event, "raw_data": joined[:500], "error": str(exc)},
                    }
    finally:
        if client is None:
            await _client.aclose()


In [ ]:
#| export
async def _async_iter_raw_and_normalized_agent_events(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
    client: httpx.AsyncClient | None = None,
) -> AsyncIterator[tuple[dict, list[dict]]]:
    """Yield ``(raw_event, normalized_events)`` tuples (async).

    Private driver that owns the ``seen_tool_result`` state machine.
    All higher-level async functions build on this.
    """
    seen_tool_result = False
    async for raw in async_stream_agent_sse(
        agent_name, question, history, session=session,
        thread_id=thread_id, parent_message_id=parent_message_id,
        client=client,
    ):
        raw_event = raw["event"]
        raw_data = raw.get("data", {})
        if raw_event == "response.tool_result":
            seen_tool_result = True
        normalized = normalize_event(raw_event, raw_data, seen_tool_result=seen_tool_result)
        yield raw, normalized


async def async_iter_normalized_agent_events(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
    client: httpx.AsyncClient | None = None,
) -> AsyncIterator[dict]:
    """Stream and normalize Cortex Agent events in one step (async).

    Yields normalized ``{"event": str, "data": dict}`` dicts. This is the
    recommended entry point for building async SSE proxies.
    """
    async for _raw, normalized_items in _async_iter_raw_and_normalized_agent_events(
        agent_name, question, history, session=session,
        thread_id=thread_id, parent_message_id=parent_message_id,
        client=client,
    ):
        for evt in normalized_items:
            yield evt


async def async_collect_agent_events(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
    client: httpx.AsyncClient | None = None,
) -> AgentResult:
    """Pure async collector: stream and accumulate into AgentResult."""
    result = AgentResult()
    start = time.time()
    current_thinking = ""

    async for raw, normalized_items in _async_iter_raw_and_normalized_agent_events(
        agent_name, question, history, session=session,
        thread_id=thread_id, parent_message_id=parent_message_id,
        client=client,
    ):
        result.raw_events.append(raw)
        for evt in normalized_items:
            current_thinking = _apply_normalized_event(
                result, evt["event"], evt["data"], current_thinking
            )

    _finalize_agent_result(result, current_thinking)
    result.duration_seconds = round(time.time() - start, 2)
    return result


async def async_run_agent(
    agent_name: str,
    question: str,
    history: list[dict] | None = None,
    verbose: bool = True,
    reporter: Callable[[str, dict], None] | None = None,
    session: SnowflakeSession | None = None,
    thread_id: str | None = None,
    parent_message_id: str | None = None,
    client: httpx.AsyncClient | None = None,
) -> AgentResult:
    """Call a Cortex Agent with async streaming SSE and accumulate the result.

    Args:
        agent_name: Name of the Cortex Agent to call.
        question: User question text.
        history: Optional conversation history.
        verbose: If True and no reporter given, prints progress to stdout.
        reporter: Optional callback ``(event_type, event_data) -> None``.
        session: Optional SnowflakeSession; uses default if None.
        thread_id: Optional Cortex thread identifier for server-side continuity.
        parent_message_id: Optional parent message for threading.
        client: Optional ``httpx.AsyncClient`` for connection reuse.
    """
    cb = reporter or (_default_reporter if verbose else None)

    if verbose and cb is _default_reporter:
        print(f"Calling {agent_name} with: {question!r}")
        print("-" * 60)

    result = AgentResult()
    start = time.time()
    current_thinking = ""

    async for raw, normalized_items in _async_iter_raw_and_normalized_agent_events(
        agent_name, question, history, session=session,
        thread_id=thread_id, parent_message_id=parent_message_id,
        client=client,
    ):
        result.raw_events.append(raw)
        for evt in normalized_items:
            etype = evt["event"]
            edata = evt["data"]
            current_thinking = _apply_normalized_event(
                result, etype, edata, current_thinking
            )
            if cb:
                if etype == "table":
                    df = result.dataframes[-1]
                    cb(etype, {"result_set": edata, "rows": len(df), "cols": len(df.columns)})
                else:
                    cb(etype, edata)

    _finalize_agent_result(result, current_thinking)
    result.duration_seconds = round(time.time() - start, 2)

    if verbose and cb is _default_reporter:
        print("-" * 60)
        print(f"Done in {result.duration_seconds}s | Tools: {result.tools_used} | SQL: {len(result.sql_queries)} | Tables: {len(result.dataframes)}")

    return result


class AsyncAgentChat:
    """Async stateful conversation wrapper for Cortex Agents.

    Operates in two modes:

    - **Local-history mode** (default): conversation history is sent with
      each request.
    - **Thread mode** (``thread_id`` provided): the server-side thread
      owns continuity. Local history is still recorded for inspection.

    Lower-level control is available via ``async_run_agent()`` and
    ``async_stream_agent_sse()``.
    """

    def __init__(
        self,
        agent_name: str,
        session: SnowflakeSession | None = None,
        verbose: bool = True,
        reporter: Callable[[str, dict], None] | None = None,
        history: list[dict] | None = None,
        thread_id: str | None = None,
        client: httpx.AsyncClient | None = None,
    ):
        self.agent_name = agent_name
        self.session = session
        self.verbose = verbose
        self.reporter = reporter
        self.history: list[dict] = list(history or [])
        self.results: list[AgentResult] = []
        self.thread_id = thread_id
        self.client = client
        self._parent_message_id: str | None = None

    async def ask(self, question: str, **kwargs) -> AgentResult:
        """Send a question; history/thread state is carried automatically."""
        result = await async_run_agent(
            self.agent_name,
            question,
            history=self.history if not self.thread_id else None,
            verbose=kwargs.get("verbose", self.verbose),
            reporter=kwargs.get("reporter", self.reporter),
            session=self.session,
            thread_id=self.thread_id,
            parent_message_id=self._parent_message_id,
            client=self.client,
        )
        if result.thread_metadata.get("message_id"):
            self._parent_message_id = result.thread_metadata["message_id"]
        self.history.append({"role": "user", "content": question})
        assistant_text = result.answer or "\n".join(result.thinking)
        self.history.append({"role": "assistant", "content": assistant_text})
        self.results.append(result)
        return result

    @property
    def last(self) -> AgentResult | None:
        """Most recent result, or None if no questions asked yet."""
        return self.results[-1] if self.results else None

    def reset(self) -> "AsyncAgentChat":
        """Clear local transcript, results, and parent message cursor.

        Does **not** unset ``thread_id``.
        """
        self.history.clear()
        self.results.clear()
        self._parent_message_id = None
        return self

    def __repr__(self) -> str:
        mode = f"thread={self.thread_id}" if self.thread_id else "local-history"
        return f"AsyncAgentChat({self.agent_name!r}, turns={len(self.results)}, {mode})"


In [ ]:
import inspect
from mcp_ski_resort.core import (
    stream_agent_sse, iter_normalized_agent_events, run_agent,
    collect_agent_events, create_thread, AgentChat,
)

sync_async_pairs = [
    (stream_agent_sse, async_stream_agent_sse),
    (iter_normalized_agent_events, async_iter_normalized_agent_events),
    (run_agent, async_run_agent),
    (collect_agent_events, async_collect_agent_events),
    (create_thread, async_create_thread),
]

for sync_fn, async_fn in sync_async_pairs:
    sync_params = set(inspect.signature(sync_fn).parameters.keys())
    async_params = set(inspect.signature(async_fn).parameters.keys())
    extra = async_params - sync_params
    assert extra <= {"client"}, f"{async_fn.__name__} has unexpected extra params: {extra - {'client'}}"
    missing = sync_params - async_params
    assert not missing, f"{async_fn.__name__} missing params: {missing}"
    print(f"  {sync_fn.__name__} <-> {async_fn.__name__}: OK")

for method in ["ask", "reset", "last"]:
    assert hasattr(AsyncAgentChat, method), f"AsyncAgentChat missing {method}"
print("  AsyncAgentChat API parity: OK")

ac = AsyncAgentChat("test", thread_id="t-1")
assert "thread=t-1" in repr(ac)
ac.reset()
assert ac.thread_id == "t-1"
print("  AsyncAgentChat reset keeps thread_id: OK")

print("\nAll async parity tests passed")


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()